In [1]:
from jax import numpy as jnp
from flax import nnx
import optax
from tqdm import tqdm

In [2]:
class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, d_head: int, n_heads: int, rngs: nnx.Rngs):
        self.d_head = d_head
        l_n = rngs.lecun_normal(batch_axis=(0,))
        self.W_Q = nnx.Param(l_n((n_heads, d_model, d_head)))
        self.W_K = nnx.Param(l_n((n_heads, d_model, d_head)))
        self.W_V = nnx.Param(l_n((n_heads, d_model, d_head)))
        self.W_O = nnx.Param(l_n((n_heads, d_head, d_model)))

    def __call__(self, X: jnp.ndarray): # (seq_len, d_model)
        Q = jnp.einsum("nmh,sm->nsh", self.W_Q, X) # (head_n, seq_len, d_head) 
        K = jnp.einsum("nmh,sm->nsh", self.W_K, X) # (head_n, seq_len, d_head) 
        V = jnp.einsum("nmh,sm->nsh", self.W_V, X) # (head_n, seq_len, d_head) 

        scores = jnp.einsum("nqh,nkh->nqk", Q, K)/jnp.sqrt(self.d_head) # (head_n, seq_len, seq_len)
        weights = nnx.softmax(scores, axis=(2)) # (head_n, seq_len, seq_len)
        out = jnp.einsum("nqk,nkd->nqd", weights, V) # (head_n, seq_len, d_head)
        
        final = jnp.einsum("nqd,ndm->nqm", out, self.W_O) # (head_n, seq_len, d_model)
        final = final.sum(axis=0) # (seq_len, d_model)

        return final
